In [28]:
import time
import warnings
from collections import Counter
from itertools import combinations

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.linear_model import (
    ElasticNet,
    Lasso,
    LinearRegression,
    Ridge,
)
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    MinMaxScaler,
    OrdinalEncoder,
    PolynomialFeatures,
)
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

warnings.filterwarnings("ignore")

In [2]:
train_processed = pd.read_csv("train_processed.csv")
val_processed = pd.read_csv("val_processed.csv")

In [3]:
# Feature (X)
X_train = train_processed[["item_price_min", "item_price_max"]]
X_val = val_processed[["item_price_min", "item_price_max"]]

# Target (y)
y_train = train_processed["price"]
y_val = val_processed["price"]

In [4]:
print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)

(205848, 2) (205848,)
(12210, 2) (12210,)


### Use Original Price

In [5]:
model = LinearRegression()
model.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [6]:
y_pred = model.predict(X_val)

In [7]:
# Metrics
mae = mean_absolute_error(y_val, y_pred)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
mape = mean_absolute_percentage_error(y_val, y_pred) * 100  

print(f"MAE : {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAPE: {mape:.2f}%")

MAE : 40.6421
RMSE: 117.2040
MAPE: 16.25%


### Use Log1p Price

In [8]:
X_train_log = np.log1p(X_train)
X_val_log = np.log1p(X_val)

In [9]:
y_train_log = np.log1p(y_train)

model = LinearRegression()
model.fit(X_train_log, y_train_log)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [10]:
y_pred_log = model.predict(X_val_log)
y_pred = np.expm1(y_pred_log)

In [11]:
# Metrics
mae = mean_absolute_error(y_val, y_pred)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
mape = mean_absolute_percentage_error(y_val, y_pred) * 100  

print(f"MAE : {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAPE: {mape:.2f}%")

MAE : 39.6355
RMSE: 123.1752
MAPE: 14.13%


### Use All Features

In [12]:
train_processed = pd.read_csv("train_processed.csv")
val_processed = pd.read_csv("val_processed.csv")

categorical_features = ["shopId", "itemId", "modelId", "cat_id", "brand", "promotionId"]
price_features = ['priceBeforeDiscount', 'item_price_min', 'item_price_max']

# Drop categorical features
train_processed = train_processed.drop(columns=categorical_features)
val_processed = val_processed.drop(columns=categorical_features)

# Log transform price features
train_processed[price_features] = np.log1p(train_processed[price_features])
val_processed[price_features] = np.log1p(val_processed[price_features])

# Feature (X)
X_train = train_processed.drop(columns=["price"])
X_val = val_processed.drop(columns=["price"])

# Target (y)
y_train = train_processed["price"]
y_val = val_processed["price"]

In [13]:
y_train_log = np.log1p(y_train)

model = LinearRegression()
model.fit(X_train, y_train_log)

y_pred_log = model.predict(X_val)
y_pred = np.expm1(y_pred_log)

# Metrics
mae = mean_absolute_error(y_val, y_pred)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
mape = mean_absolute_percentage_error(y_val, y_pred) * 100  

print(f"MAE : {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAPE: {mape:.2f}%")

MAE : 41.8138
RMSE: 123.9430
MAPE: 14.69%


In [14]:
coef_df = pd.DataFrame({
    "feature": X_train.columns,
    "coef": model.coef_,
    "abs_coef": np.abs(model.coef_)
})

coef_df = coef_df.sort_values("abs_coef", ascending=False)
coef_df

,feature,coef,abs_coef
7,total_rating_count,-17.332048,17.332048
8,cmt_count,17.104216,17.104216
4,item_price_min,0.530085,0.530085
5,item_price_max,0.463768,0.463768
6,review_rating,0.042597,0.042597
3,is_pre_order,0.025116,0.025116
11,shop_follower_count,0.021030,0.021030
10,shop_response_rate,-0.015286,0.015286
9,shop_rating,0.007955,0.007955
12,is_official_shop,-0.007567,0.007567


### Use Other Models

In [15]:
train_processed = pd.read_csv("train_processed.csv")
val_processed = pd.read_csv("val_processed.csv")

categorical_features = ["shopId", "itemId", "modelId", "cat_id", "brand", "promotionId"]
price_features = ['priceBeforeDiscount', 'item_price_min', 'item_price_max']

# Drop categorical features
train_processed = train_processed.drop(columns=categorical_features)
val_processed = val_processed.drop(columns=categorical_features)

# Log transform price features
train_processed[price_features] = np.log1p(train_processed[price_features])
val_processed[price_features] = np.log1p(val_processed[price_features])

# Feature (X)
X_train = train_processed.drop(columns=["price"])
X_val = val_processed.drop(columns=["price"])

# Target (y)
y_train = train_processed["price"]
y_val = val_processed["price"]

In [21]:
y_train_log = np.log1p(y_train)

results = []

models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(random_state=42),
    "Lasso": Lasso(random_state=42),
    "ElasticNet": ElasticNet(random_state=42),
    "DecisionTree": DecisionTreeRegressor(
        max_depth=40,
        random_state=42
    ),
    "RandomForest": RandomForestRegressor(
        n_estimators=100,
        max_depth=20,
        random_state=42,
        n_jobs=-1
    ),
    "ExtraTrees": ExtraTreesRegressor(
        n_estimators=100,
        max_depth=20,
        random_state=42,
        n_jobs=-1
    ),
    "KNeighbors": KNeighborsRegressor(),
}

for name, model in models.items():
    # Measure training time
    start = time.perf_counter()
    model.fit(X_train, y_train_log)
    train_time = time.perf_counter() - start

    y_pred_log = model.predict(X_val)
    y_pred = np.expm1(y_pred_log)

    # Metrics
    mae = mean_absolute_error(y_val, y_pred)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mape = mean_absolute_percentage_error(y_val, y_pred) * 100

    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "MAPE (%)": mape,
        "Train Time (s)": train_time,
    })

In [22]:
results_df = pd.DataFrame(results).sort_values("MAE")

In [23]:
results_df

,Model,MAE,RMSE,MAPE (%),Train Time (s)
4,DecisionTree,24.408242,99.115346,7.241181,1.394638
6,ExtraTrees,24.560883,99.342866,7.347468,12.409723
5,RandomForest,24.591608,100.319624,7.296244,23.490808
1,Ridge,41.414903,123.756240,14.504669,0.040292
7,KNeighbors,41.463737,178.056791,10.136387,0.022443
0,LinearRegression,41.813803,123.942983,14.692445,0.103904
3,ElasticNet,216.071274,735.773873,59.078913,0.051965
2,Lasso,280.908779,865.026829,105.030375,0.126461


### Polynomial Features

In [24]:
train_processed = pd.read_csv("train_processed.csv")
val_processed = pd.read_csv("val_processed.csv")

categorical_features = ["shopId", "itemId", "modelId", "cat_id", "brand", "promotionId"]
price_features = ['priceBeforeDiscount', 'item_price_min', 'item_price_max']

# Drop categorical features
train_processed = train_processed.drop(columns=categorical_features)
val_processed = val_processed.drop(columns=categorical_features)

# Log transform price features
train_processed[price_features] = np.log1p(train_processed[price_features])
val_processed[price_features] = np.log1p(val_processed[price_features])

# Feature (X)
X_train = train_processed.drop(columns=["price"])
X_val = val_processed.drop(columns=["price"])

# Target (y)
y_train = train_processed["price"]
y_val = val_processed["price"]

In [26]:
# Log transform target
y_train_log = np.log1p(y_train)

# Polynomial Regression
model = Pipeline([
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("lr", LinearRegression())
])

model.fit(X_train, y_train_log)

# Prediction
y_pred_log = model.predict(X_val)
y_pred = np.expm1(y_pred_log)

# Metrics
mae = mean_absolute_error(y_val, y_pred)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
mape = mean_absolute_percentage_error(y_val, y_pred) * 100

print(f"MAE : {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAPE: {mape:.2f}%")

MAE : 38.0630
RMSE: 115.5438
MAPE: 11.66%


In [30]:
lr = model.named_steps["lr"]

feature_names = model.named_steps["poly"].get_feature_names_out(X_train.columns)
coef_df = pd.DataFrame({
    "feature": feature_names,
    "coef": lr.coef_,
    "abs_coef": np.abs(lr.coef_)
})

coef_df = coef_df.sort_values("abs_coef", ascending=False)
coef_df.head(20)

,feature,coef,abs_coef
116,total_rating_count cmt_count,-4010.281035,4010.281035
125,cmt_count^2,2043.436123,2043.436123
115,total_rating_count^2,1961.099902,1961.099902
8,cmt_count,1500.689178,1500.689178
106,review_rating cmt_count,-1432.295683,1432.295683
7,total_rating_count,-1289.788503,1289.788503
105,review_rating total_rating_count,1215.882171,1215.882171
128,cmt_count shop_follower_count,-223.796238,223.796238
119,total_rating_count shop_follower_count,215.141817,215.141817
129,cmt_count is_official_shop,159.149706,159.149706
